# 🚗 Cars Dataset — Exploratory Data Analysis

**Portfolio project | Python · Pandas · NumPy · Matplotlib · Seaborn**

## Objective
Explore a car dataset to understand data quality, MSRP distribution, manufacturer-level price patterns, and relationships between vehicle characteristics. This is a descriptive EDA project, not a prediction model.

## Business questions
1. What are the dataset's dimensions, columns, and data types?
2. Which fields contain missing values or duplicate records?
3. How does MSRP vary across manufacturers?
4. What patterns appear in engine horsepower and fuel economy?
5. Which numeric features are correlated with MSRP?

> Run the notebook from top to bottom. Review the outputs and write your own verified findings before presenting this project to recruiters.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 2. Load the dataset
Keep `Cars_data_.csv` in the same folder as this notebook.

In [ ]:
possible_paths = [Path('Cars_data_.csv'), Path('./data/Cars_data_.csv'), Path('/content/Cars_data_.csv')]
csv_path = next((p for p in possible_paths if p.exists()), None)

if csv_path is None:
    raise FileNotFoundError('Cars_data_.csv not found. Put it beside this notebook or update possible_paths.')

df_original = pd.read_csv(csv_path)
df = df_original.copy()
print(f'Loaded: {csv_path}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]:,}')
display(df.head())

## 3. Understand the dataset

In [ ]:
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nData types and non-null counts:')
df.info()
print('\nNumerical summary:')
display(df.describe(include='number').T)
print('\nCategorical summary:')
display(df.describe(include='object').T)

## 4. Check data quality
Measure missing values and exact duplicate rows before deciding how to clean the data.

In [ ]:
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2)
}).sort_values('missing_percent', ascending=False)
print('Exact duplicate rows:', int(df.duplicated().sum()))
display(missing[missing['missing_count'] > 0])

missing_plot = missing.loc[missing['missing_count'] > 0].sort_values('missing_percent')
if not missing_plot.empty:
    plt.figure(figsize=(10, 5))
    plt.barh(missing_plot.index, missing_plot['missing_percent'])
    plt.xlabel('Missing values (%)')
    plt.title('Missing-value percentage by column')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values found.')

## 5. Clean the data
Standardize whitespace, remove exact duplicate rows, and convert commonly numeric fields when present. Missing values are not filled with arbitrary values.

In [ ]:
df.columns = df.columns.astype(str).str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].map(lambda value: value.strip() if isinstance(value, str) else value)

rows_before = len(df)
df = df.drop_duplicates().copy()
duplicates_removed = rows_before - len(df)

numeric_candidates = ['Year', 'Engine HP', 'Engine Cylinders', 'Number of Doors', 'highway MPG', 'city mpg', 'Popularity', 'MSRP']
for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Duplicate rows removed: {duplicates_removed:,}')
print(f'Cleaned shape: {df.shape}')
display(df.head())
display(df.isna().sum().loc[lambda s: s.gt(0)].rename('missing_count').to_frame())

## 6. Analyze MSRP
MSRP-specific analysis excludes rows with missing or non-positive MSRP. Those rows remain in `df` for other analyses.

In [ ]:
if 'MSRP' not in df.columns:
    raise KeyError('Expected an MSRP column. Review the actual dataset columns and update the notebook.')

price_df = df.loc[df['MSRP'].notna() & (df['MSRP'] > 0)].copy()
print(f'Rows available for MSRP analysis: {len(price_df):,} of {len(df):,}')
display(price_df['MSRP'].describe().to_frame('MSRP'))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(price_df['MSRP'], bins=40, kde=True, ax=axes[0])
axes[0].set(title='Distribution of vehicle MSRP', xlabel='MSRP', ylabel='Count')
sns.boxplot(x=price_df['MSRP'], ax=axes[1])
axes[1].set(title='MSRP box plot', xlabel='MSRP')
plt.tight_layout()
plt.show()

## 7. Compare manufacturers
Compare manufacturers with the most records. Median MSRP is included because the mean can be strongly affected by expensive models.

In [ ]:
if 'Make' in price_df.columns:
    make_summary = price_df.groupby('Make')['MSRP'].agg(
        vehicle_count='count', mean_msrp='mean', median_msrp='median'
    ).sort_values('median_msrp', ascending=False)
    display(make_summary.head(15))

    common_makes = price_df['Make'].value_counts().head(12).index
    plot_df = price_df[price_df['Make'].isin(common_makes)]
    plt.figure(figsize=(12, 5))
    sns.barplot(data=plot_df, x='Make', y='MSRP', estimator='median', errorbar=None)
    plt.title('Median MSRP among the 12 most represented manufacturers')
    plt.xlabel('Manufacturer')
    plt.ylabel('Median MSRP')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No Make column found; skipping manufacturer comparison.')

## 8. Explore vehicle characteristics

In [ ]:
features = [c for c in ['Engine HP', 'Year', 'city mpg', 'highway MPG'] if c in df.columns]
if features:
    fig, axes = plt.subplots(len(features), 1, figsize=(9, 3.2 * len(features)))
    axes = np.atleast_1d(axes)
    for ax, col in zip(axes, features):
        values = df[col].dropna()
        if len(values):
            sns.histplot(values, bins=30, kde=True, ax=ax)
            ax.set_title(f'Distribution of {col}')
            ax.set_xlabel(col)
    plt.tight_layout()
    plt.show()
else:
    print('No expected vehicle characteristic columns found.')

## 9. Correlation analysis
Correlation describes linear association, not causation. Results can be affected by missing values, outliers, and the mix of vehicles in the dataset.

In [ ]:
numeric_cols = [c for c in ['MSRP', 'Year', 'Engine HP', 'Engine Cylinders', 'Number of Doors', 'highway MPG', 'city mpg', 'Popularity'] if c in price_df.columns and pd.api.types.is_numeric_dtype(price_df[c])]
if len(numeric_cols) >= 2:
    corr = price_df[numeric_cols].corr(numeric_only=True)
    display(corr)
    plt.figure(figsize=(9, 6))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
    plt.title('Correlation matrix for numeric features')
    plt.tight_layout()
    plt.show()
else:
    print('Not enough numeric columns for correlation analysis.')

if 'Engine HP' in price_df.columns:
    scatter = price_df[['Engine HP', 'MSRP']].dropna()
    if not scatter.empty:
        plt.figure(figsize=(8, 5))
        sns.scatterplot(data=scatter, x='Engine HP', y='MSRP', alpha=.4)
        plt.title('Engine horsepower vs MSRP')
        plt.tight_layout()
        plt.show()

## 10. Generate a data-driven summary
This section calculates results from the loaded file. Check the outputs and use only verified findings in your portfolio or interviews.

In [ ]:
print('DATASET SUMMARY')
print('-' * 40)
print(f'Rows after duplicate removal: {len(df):,}')
print(f'Columns: {df.shape[1]:,}')
print(f'Duplicate rows removed: {duplicates_removed:,}')
print(f'Columns with missing values: {(df.isna().sum() > 0).sum():,}')

if not price_df.empty:
    print('\nMSRP SUMMARY')
    print(f'Median MSRP: {price_df["MSRP"].median():,.2f}')
    print(f'Mean MSRP: {price_df["MSRP"].mean():,.2f}')
    print(f'Minimum positive MSRP: {price_df["MSRP"].min():,.2f}')
    print(f'Maximum MSRP: {price_df["MSRP"].max():,.2f}')

if 'Make' in price_df.columns and not price_df.empty:
    counts = price_df['Make'].value_counts()
    medians = price_df.groupby('Make')['MSRP'].median().sort_values(ascending=False)
    print(f'\nManufacturers represented: {price_df["Make"].nunique():,}')
    print(f'Most represented manufacturer: {counts.index[0]} ({counts.iloc[0]:,} rows)')
    print(f'Highest median MSRP manufacturer: {medians.index[0]} ({medians.iloc[0]:,.2f})')

if {'Engine HP', 'MSRP'}.issubset(price_df.columns):
    paired = price_df[['Engine HP', 'MSRP']].dropna()
    if len(paired) >= 2:
        print(f'\nHorsepower/MSRP correlation: {paired.corr().iloc[0, 1]:.2f}')

## 11. Conclusions and limitations

Use the calculated outputs above to write 3–5 specific findings in your own words. Avoid adding results that were not produced by the notebook.

### Possible next steps
- Segment price analysis by vehicle size, style, model year, or fuel type.
- Investigate missing values and extreme records with domain knowledge.
- Build a dashboard to make comparisons interactive.
- Create a separate MSRP prediction project only after defining a train/test split and suitable evaluation metrics.

### Limitations
- The dataset may not represent the current vehicle market.
- MSRP is not necessarily the final sale price.
- Outliers are not automatically data errors.
- Correlation does not establish causation.
- This notebook performs descriptive analysis and does not train a predictive model.

## Skills demonstrated
`Python` · `Pandas` · `NumPy` · `Data Cleaning` · `EDA` · `Data Visualization` · `Correlation Analysis`

## How to run
1. Keep `Cars_data_.csv` in the same folder as this notebook.
2. Install dependencies: `pip install pandas numpy matplotlib seaborn jupyter`
3. Run `jupyter notebook` and execute every cell in order.

You can also open the notebook in Google Colab and upload the CSV file.